# Scaling up to large datasets

<a target="_blank" href="https://colab.research.google.com/github/RobinL/splink/blob/ipynbs/docs/demos/tutorials/09_scaling_up_techniques.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

The previous tutorials showed how to build, estimate and use a linkage model. On small to medium sized data, the default settings 'just work' and you don't need to do anything special.

As your input data grows to tens or hundreds of millions of records, some steps can become slow to run, or you may run out of memory. This tutorial introduces the tools Splink provides to ensure you can still run linkages even on very large datasets.

1. **Model training** — using sampling to speed up blocking analysis and parameter estimation.
2. **Inference** — using chunking to control the memory used by `predict()`, and to distribute the work across multiple processes or machines.

!!! warning "Only reach for these techniques when you need them"
    The techniques on this page trade something away — either accuracy (sampling) or simplicity (chunking). There is no need to use any of them if your job is already running quickly. Start with the defaults, and only introduce sampling or chunking at the point where a step becomes too slow or runs out of memory.

To keep this tutorial fast to run, we use the small `fake_1000` dataset. In reality you would never need these techniques on data this small — the examples are here purely to demonstrate the API. As you read, imagine the same code running on a dataset that has hundreds of millions of rows.

In [1]:
# Uncomment and run this cell if you're running in Google Colab.
# !pip install "splink[altair,igraph,pyarrow] @ git+https://github.com/RobinL/splink.git@master"

In [2]:
import splink.comparison_library as cl
from splink import (
    ColumnExpression,
    DuckDBAPI,
    Linker,
    SettingsCreator,
    block_on,
    splink_datasets,
)

df = splink_datasets.fake_1000

db_api = DuckDBAPI()
df_sdf = db_api.register(df)

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("first_name"),
        cl.NameComparison("surname"),
        cl.LevenshteinAtThresholds(ColumnExpression("dob").cast_to_string(), 1),
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        cl.EmailComparison("email"),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("first_name", "city"),
        block_on("surname"),
        block_on("dob"),
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df_sdf, settings)

## Section 1: Blocking analysis and model training

### Sampling in blocking rule analysis

In [tutorial 3](./03_Blocking.ipynb) we used `count_comparisons_from_blocking_rules` and `chart_comparisons_from_blocking_rules` to count how many comparisons our blocking rules generate.

Computing these counts exactly means executing the blocking join over the whole dataset, which can be expensive on large data. To avoid this, both functions accept a `record_sample_proportion` argument. Splink takes a deterministic sample of the input records on each side of the blocking join, counts the comparisons generated by the sample, and scales the result back up to estimate the full count.

`record_sample_proportion` defaults to `0.05` (a 5% sample), so these functions already estimate from a sample by default. Lowering it makes the estimate faster but less precise; setting `record_sample_proportion=1.0` computes the exact count.

!!! note "Sampling is quadratically cheaper"
    Blocking is a join, so the number of comparisons grows quadratically with the number of records. Because `record_sample_proportion` samples both sides of the join, taking a fraction `p` of records causes a reduction in comparisons of `p²`. Splink then scales the count back up by `1 / p²`). So a 10% sample (`record_sample_proportion=0.1`) generates roughly 1% of the comparisons.

In [3]:
from splink.blocking_analysis import count_comparisons_from_blocking_rules

# Estimate the count from a 30% sample of records (faster, approximate)
counts_estimated = count_comparisons_from_blocking_rules(
    df_sdf,
    blocking_rules=block_on("first_name", "city"),
    link_type="dedupe_only",
    record_sample_proportion=0.3,
)
estimated_count = counts_estimated[0]["marginal_comparison_count"]
print(f"Estimated from 30% sample: {estimated_count:,} comparisons")

Estimated from 30% sample: 356 comparisons


/home/runner/work/splink/splink/splink/internals/blocking_analysis.py:668: UserWarning: The sampled blocking analysis estimate for blocking rule '(l."first_name" = r."first_name") AND (l."city" = r."city")' is based on 32 sampled pairwise comparisons. This is below the recommended minimum of 1,000, so the estimate may be unstable. Increase record_sample_proportion for a more stable estimate.
  return _cumulative_comparisons_to_be_scored_from_blocking_rules(


In [4]:
# Compute the exact count over all records (slower, exact)
counts_exact = count_comparisons_from_blocking_rules(
    df_sdf,
    blocking_rules=block_on("first_name", "city"),
    link_type="dedupe_only",
    record_sample_proportion=1.0,
)
exact_count = counts_exact[0]["marginal_comparison_count"]
print(f"Exact over all records: {exact_count:,} comparisons")

Exact over all records: 315 comparisons


On a small dataset like this the sampled estimate may differ noticeably from the exact count — there simply aren't many records to sample from. On large datasets a small sample gives a good estimate, which is exactly where the speed-up matters.

The same `record_sample_proportion` argument is available on `chart_comparisons_from_blocking_rules`.

### Sampling when estimating `probability_two_random_records_match`

`estimate_probability_two_random_records_match` works by counting how many pairs are matched by a set of deterministic rules. On large data, counting these matches exactly can be slow.

The method accepts the same `record_sample_proportion` argument, which samples each side of the deterministic blocking join and scales the observed match count back up. It defaults to `1.0` (exact). Lower it on large data to trade a little accuracy for speed.

In [5]:
deterministic_rules = [
    block_on("first_name", "dob"),
    "l.first_name = r.first_name and levenshtein(r.surname, l.surname) <= 2",
    block_on("email"),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules,
    recall=0.7,
    record_sample_proportion=0.5,
)

/home/runner/work/splink/splink/splink/internals/linker_components/training.py:102: UserWarning: The sampled blocking analysis estimate for blocking rule '(l."first_name" = r."first_name") AND (l."dob" = r."dob")' is based on 57 sampled pairwise comparisons. This is below the recommended minimum of 1,000, so the estimate may be unstable. Increase record_sample_proportion for a more stable estimate.
  records = _cumulative_comparisons_to_be_scored_from_blocking_rules(
/home/runner/work/splink/splink/splink/internals/linker_components/training.py:102: UserWarning: The sampled blocking analysis estimate for blocking rule 'l.first_name = r.first_name and levenshtein(r.surname, l.surname) <= 2' is based on 66 sampled pairwise comparisons. This is below the recommended minimum of 1,000, so the estimate may be unstable. Increase record_sample_proportion for a more stable estimate.
  records = _cumulative_comparisons_to_be_scored_from_blocking_rules(
/home/runner/work/splink/splink/splink/inte

### Controlling the cost of `estimate_u_using_random_sampling` with `max_pairs`

`estimate_u_using_random_sampling` estimates the `u` probabilities by sampling random pairs of records. The `max_pairs` argument controls how many pairs are sampled and is the main lever on both runtime and accuracy:

- Larger `max_pairs` → more accurate `u` estimates, but longer runtime.
- Smaller `max_pairs` → faster, but noisier estimates.

While iterating on a model specification, a smaller value such as `1e6` or `1e7` keeps the feedback loop fast. For a final model we recommend `1e8` or even `1e9`.
Importantly, note that for a constant `max_pairs`  `estimate_u_using_random_sampling` will run at a similar speed irrespective of the size of the dataset, since it's the number of comparison pairs which is the main determinant of the runtime for this function.

In [6]:
linker.training.estimate_u_using_random_sampling(max_pairs=2e6)

----- Estimating u probabilities using random sampling -----


Estimating u with: max_pairs = 2,000,000, min_count_per_level = 100, num_chunks = 10



Estimating u for: first_name (Comparison 1 of 5)


  Running probe chunk (~1.00% of max_pairs)


  Min u_count: 7 for comparison level Jaro-Winkler distance of first_name >= 0.88 (cvv=2)


...

  Running chunk 6/10


  Count of 113 for level Jaro-Winkler >0.88 on username (cvv=1). Chunk took 0.1 seconds.


  Exiting early since min count of 113 exceeds min_count_per_level = 100



Estimated u probabilities using random sampling



Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


### Sampling in Expectation Maximisation with `max_pairs`

`estimate_parameters_using_expectation_maximisation` generates pairwise comparisons using the training blocking rule and iterates over them. On large data, a training rule can still generate a very large number of pairs, making each EM iteration slow.

Two arguments let you cap this cost:

- **`max_pairs`** (default `None`): if set, Splink limits the approximate number of blocked pairs used for EM training to this value. It runs a quick preliminary blocking pass to estimate the full blocked-pair count, then applies a deterministic filter to the input records so that the resulting number of blocked pairs is approximately `max_pairs`. With the default of `None`, all blocked pairs are used (no sampling).
- **`record_sample_proportion`** (default `0.01`): the fraction of input records sampled on each side of that preliminary pass used to *estimate* the full blocked-pair count. It does not directly set the final number of training pairs — `max_pairs` remains the primary control.

In other words, set `max_pairs` to cap how much data each EM training session uses. Here we pass a deliberately high `max_pairs` so the small `fake_1000` model is not actually downsampled.
An important caveat is that for EM training to work, your pairs must contain a reasonable number of matches.  If you choose a very loose blocking rule like `block_on("gender")` on a large dataset, even a large `max_pairs` value may result in very few true matches, and the training is unlikely to succeed.

In [7]:
training_blocking_rule = block_on("dob")
linker.training.estimate_parameters_using_expectation_maximisation(
    training_blocking_rule,
    max_pairs=1e7
)


----- Starting EM training session -----



[EM sampling] Probe at proportion 0.010000 (actual fraction 0.010000, sample_threshold=100 / sample_modulus=10,000) -> 0 blocked pairs


[EM sampling] Probe returned zero blocked pairs; skipping sampling


Estimating the m probabilities of the model by blocking on:
l."dob" = r."dob"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - surname
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - dob


Level Jaro-Winkler >0.88 on username on comparison email not observed in dataset, unable to train m value



...

Iteration 13: Largest change in params was 5.06e-05 in probability_two_random_records_match



EM converged after 13 iterations


m probability not trained for email - Jaro-Winkler >0.88 on username (comparison vector value: 1). This usually means the comparison level was never observed in the training data.



Your model is not yet fully trained. Missing estimates for:
    - dob (no m values are trained).
    - email (some m values are not trained).


<EMTrainingSession, blocking on l."dob" = r."dob", deactivating comparisons dob>

We now have a trained model, which we will use to demonstrate the inference techniques below.

## Section 2: Scaling inference with chunking

`predict()` generates every blocked pair and scores it in a single pass. On very large datasets, this may be time consuming, or you may even run into out of memory or out of disk errors. Splink provides a family of chunking tools that split this work into smaller pieces.
By 'chunking' we mean that Splink allows you to compute a fixed proportion of the result.  This allows you to complete your linkage in `n` smaller pieces, which, when combined represent the full result.


The pieces are defined by a grid: the left side of each comparison is split into `num_chunks_left` slices and the right side into `num_chunks_right` slices, giving `num_chunks_left × num_chunks_right` chunks in total. Every blocked pair falls into exactly one chunk, so processing all chunks produces exactly the same result as a single `predict()` call.

### Chunked `predict()` with `num_chunks_left` and `num_chunks_right`

The simplest way to use chunking is to pass `num_chunks_left` and `num_chunks_right` to `predict()`. Splink then processes the chunks one after another (in series) and unions the results for you, so the return value is identical to an unchunked `predict()`.

This gives two benefits on large jobs:

1. **Lower peak memory** — each chunk materialises only a fraction of the blocked pairs at a time.
2. **Progress reporting** — because the chunks run in series, Splink logs progress after each one. With the default logging level (`INFO`) you will see messages such as:

   ```
   Processing chunk (1, 4) x (1, 4) [1/16]
   Completed chunk 1/16 (6%) | Elapsed: 60.0s | Remaining: ~900.0s | Total: ~960.0s
   Processing chunk (1, 4) x (2, 4) [2/16]
   ...
   ```

   This makes it possible to estimate how long a long-running job will take, which you don't get from a single opaque `predict()` call.

In [8]:
predictions_chunked = linker.inference.predict(
    threshold_match_probability=0.9,
    num_chunks_left=4,
    num_chunks_right=4,
)

# The result is identical to an unchunked predict()
predictions_single = linker.inference.predict(threshold_match_probability=0.9)

chunked_count = predictions_chunked.as_duckdbpyrelation().count("*").fetchone()[0]
single_count = predictions_single.as_duckdbpyrelation().count("*").fetchone()[0]
print(f"Chunked predict produced {chunked_count} rows")
print(f"Single  predict produced {single_count} rows")

Processing chunk (1, 4) x (1, 4) [1/16]


Blocking time: 0.01 seconds


Predict time (post-blocking): 0.07 seconds


Completed chunk 1/16 (6%) | Elapsed: 0.1s | Remaining: ~1.2s | Total: ~1.3s


Processing chunk (1, 4) x (2, 4) [2/16]


...


 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'dob':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


Blocking time: 0.00 seconds


Predict time (post-blocking): 0.02 seconds



 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'dob':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


Chunked predict produced 989 rows
Single  predict produced 989 rows


### Verifying your pipeline with `predict_chunk()`

Before committing to a long full run, it's useful to confirm that a single chunk works end to end. `predict_chunk()` computes and scores just one slice of the grid.
In the below (1,4) means 'compute chunk 1 of of 4' on the left side, and 'compute chunk 1 of 4' on the right side.

In [9]:
single_chunk = linker.inference.predict_chunk(
    left_chunk=(1, 4),
    right_chunk=(1, 4),
    threshold_match_probability=0.9,
)
single_chunk.as_duckdbpyrelation().limit(5).show(max_width=10000)

Blocking time: 0.00 seconds


Predict time (post-blocking): 0.04 seconds



 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'dob':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


┌────────────────────┬────────────────────┬─────────────┬─────────────┬──────────────┬──────────────┬──────────────────┬───────────────────────┬───────────────────────┬────────────────────┬──────────────────────┬───────────┬───────────┬───────────────┬──────────────────────┬──────────────────────┬───────────────────┬──────────────────────┬────────────┬────────────┬───────────┬───────────────────┬───────────┬───────────┬────────────┬───────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────────────┬─────────────────────────────┬─────────────┬───────────────────────┬───────────────────────┬───────────────────┬─────────────────────┬───────────┐
│    match_weight    │ match_probability  │ unique_id_l │ unique_id_r │ first_name_l │ first_name_r │ gamma_first_name │    tf_first_name_l    │    tf_first_name_r    │   mw_first_name    │ mw_tf_adj_first_name │ surname_l │ surname_r │ gamma_surname │     tf_surname_l     │     tf_surname_r    

This is a good way to de-risk a large job: if a single chunk runs successfully, you know the whole pipeline works, and you can simply iterate over every `(left_chunk, right_chunk)` combination to produce the complete result set.

Running that loop yourself and concatenating the outputs is exactly what `predict(num_chunks_left=..., num_chunks_right=...)` does for you — so for a job that runs on a single machine, prefer the chunked `predict()` shown above. The value of `predict_chunk()` is that each chunk is fully independent, which is what makes the distributed workflow in the next section possible.

### Distributing across multiple machines with `predict_chunk()`

Because every `predict_chunk()` call is completely self-contained, you can score different chunks on different machines — even with DuckDB, which is a single-node engine (i.e. runs on a single computer, not a cluster). This is how you scale a single linkage job horizontally: each machine loads the model and input data, scores its assigned chunk(s) with `predict_chunk()`, and writes the result to shared storage. A coordinator then unions the per-chunk outputs into the final result.

Note there is no automatic scheduler — you have to trigger the per-chunk jobs and combine their outputs yourself.

A typical distributed workflow looks like this:

1. **Coordinator**: save the trained model to shared storage so that every worker loads an identical model.
2. **Workers**: each machine loads the model and input data, calls `predict_chunk()` for its assigned `(left_chunk, right_chunk)` slice(s), and writes the scored output to shared storage (e.g. parquet on cloud storage).
3. **Coordinator**: collect and concatenate the per-chunk prediction outputs.

Below we simulate this on a single machine. First, the coordinator saves the model so that workers can load an identical copy:

In [10]:
import os
import tempfile

work_dir = tempfile.mkdtemp()
model_path = os.path.join(work_dir, "model.json")

# The coordinator saves the trained model so every worker loads an identical model
linker.misc.save_model_to_json(model_path, overwrite=True)

{'link_type': 'dedupe_only',
 'probability_two_random_records_match': 0.002882882882882883,
 'retain_matching_columns': True,
 'retain_intermediate_calculation_columns': True,
 'additional_columns_to_retain': [],
 'sql_dialect': 'duckdb',
 'linker_uid': 'jea8iuxj',
 'em_convergence': 0.0001,
 'max_iterations': 25,
 'match_weight_column_prefix': 'mw_',
 'term_frequency_adjustment_column_prefix': 'tf_',
 'comparison_vector_value_column_prefix': 'gamma_',
 'unique_id_column_name': 'unique_id',
 'source_dataset_column_name': 'source_dataset',
 'blocking_rules_to_generate_predictions': [{'blocking_rule': '(l."first_name" = r."first_name") AND (l."city" = r."city")',
   'sql_dialect': 'duckdb'},
  {'blocking_rule': 'l."surname" = r."surname"', 'sql_dialect': 'duckdb'},
  {'blocking_rule': 'l."dob" = r."dob"', 'sql_dialect': 'duckdb'}],
 'comparisons': [{'output_column_name': 'first_name',
   'comparison_levels': [{'sql_condition': '"first_name_l" IS NULL OR "first_name_r" IS NULL',
     'lab

Now a worker — which could be a completely separate machine — loads the model and input data, scores its assigned chunk with `predict_chunk()`, and writes the result to shared storage:

In [11]:
# On a worker machine: rebuild the linker from the shared model and input data
db_api_worker = DuckDBAPI()
df_sdf_worker = db_api_worker.register(df)
linker_worker = Linker(df_sdf_worker, model_path)

# Score just this worker's assigned chunk and write the result to shared storage
chunk_path = os.path.join(work_dir, "predictions_chunk_1_1.parquet")
chunk_predictions = linker_worker.inference.predict_chunk(
    left_chunk=(1, 4),
    right_chunk=(1, 4),
    threshold_match_probability=0.9,
)
chunk_predictions.as_duckdbpyrelation().to_parquet(chunk_path)

chunk_count = chunk_predictions.as_duckdbpyrelation().count("*").fetchone()[0]
print(f"Worker scored {chunk_count} pairs and wrote them to {chunk_path}")

Blocking time: 0.00 seconds


Predict time (post-blocking): 0.05 seconds



 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained parameters will use default values.
Comparison: 'dob':
    m values not fully trained
Comparison: 'email':
    m values not fully trained


Worker scored 66 pairs and wrote them to /tmp/tmpx9b0ov5p/predictions_chunk_1_1.parquet


In a real deployment you would run that worker step once per chunk, each on its own machine, and then union the per-chunk parquet outputs to obtain the full set of predictions.

!!! note "Niche: separating blocking from scoring with `compute_blocked_pairs_for_predict_chunk()`"
    The workflow above scores a whole chunk in a single `predict_chunk()` call, which is all that most distributed jobs ever need. For the very largest jobs, you may wish to split blocking into separate jobs in your DAG, and run predictions separately.  To do so you do the following:

    - `compute_blocked_pairs_for_predict_chunk(left_chunk=..., right_chunk=...)` materialises only the blocked pairs (the candidate record-id pairs) for a chunk, without scoring them, which you persist to shared storage.
    - A worker later reads those pairs, registers them with `register_blocked_pairs_for_predict()`, and calls `predict()` to score exactly that table — no re-blocking. (Once blocked pairs are registered this way, the chunking arguments `num_chunks_left` / `num_chunks_right` and `predict_chunk()` are unavailable, because the pairs are already materialised.)

    This is a niche technique — reach for it only when scoring a single chunk with `predict_chunk()` is itself too large.

## Summary: when to use each technique

| Technique | Where | Use when |
| --- | --- | --- |
| `record_sample_proportion` | Blocking analysis, `estimate_probability_two_random_records_match` | Exact counts over the full dataset are too slow |
| `max_pairs` | `estimate_u_using_random_sampling`, `estimate_parameters_using_expectation_maximisation` | Training passes generate too many pairs to process quickly |
| `num_chunks_left` / `num_chunks_right` on `predict()` | Inference | `predict()` is running out of memory, or you want progress reporting |
| `predict_chunk()` | Inference | Verifying the pipeline on one slice, or distributing a job across machines by scoring one chunk per worker |
| `compute_blocked_pairs_for_predict_chunk()` + `register_blocked_pairs_for_predict()` | Inference | Niche: separating blocking from scoring when even a single chunk is too large to score in one pass |

Remember the guiding principle: start with the defaults and only adopt these techniques when a specific step becomes too slow or runs out of memory. Sampling costs you a little accuracy, and chunking adds operational complexity, so neither is worth it while everything is running quickly.

!!! note "Further Reading"
    :simple-readme: For more on performance and scaling, see the [Performance Topic Guides](../../topic_guides/performance/optimising_duckdb.md).

    :material-tools: For the full API, see the [Blocking analysis](../../api_docs/blocking_analysis.md), [Training](../../api_docs/training.md) and [Inference](../../api_docs/inference.md) API documentation.